# Topic Modeling (LDA)

In [1]:
import os, sys
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
for _c in (_HERE, os.path.dirname(_HERE)):
    _p = os.path.join(_c, "app")
    if os.path.isdir(_p):
        sys.path.insert(0, _p); break

import pandas as pd
from text_utils import CLEAN_CSV, MODELS_DIR, tag_aspects, tokenize

df = pd.read_csv(CLEAN_CSV)

# Rule-based aspects — THIS is what the app shows as "Predicted topic".
# (LDA cannot learn from 16-word reviews; this can, and it never says "Unknown".)
tagged = df["Review text"].astype(str).apply(tag_aspects)
df["aspect"] = [t[0] for t in tagged]
df["aspect_conf"] = [round(t[2], 2) for t in tagged]

print(df["aspect"].value_counts())
print("unmatched ('General'): %.1f%%" % ((df["aspect"] == "General").mean() * 100))

df.to_csv(CLEAN_CSV, index=False)
print("aspect column saved to", CLEAN_CSV)

aspect
Food        390
Service      83
General      80
Price        20
Ambience     14
Name: count, dtype: int64
unmatched ('General'): 13.6%
aspect column saved to C:\Users\HP\Desktop\Local_Business_Review\Data\processed\cleaned_restaurant_reviews.csv


In [2]:
# LDA kept ONLY for the report as unsupervised exploration
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))

texts = [[w for w in tokenize(str(t))
          if w not in STOP and len(w) > 2 and not w.startswith("neg_")]
         for t in df["Review text"].tolist()]

dictionary = corpora.Dictionary(texts)
print("vocab before filter:", len(dictionary))
dictionary.filter_extremes(no_below=2, no_above=0.6)
print("vocab after  filter:", len(dictionary))
corpus = [dictionary.doc2bow(t) for t in texts]
print("empty documents:", sum(1 for c in corpus if not c))

scores = {}
for k in [3, 4, 5, 6]:
    m = gensim.models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=k,
                               random_state=42, passes=15, alpha="auto")
    cm = CoherenceModel(model=m, texts=texts, dictionary=dictionary,
                        coherence="c_v", processes=1)      # ← 1, avoids Windows worker crash
    scores[k] = cm.get_coherence()
    print(f"k={k}  coherence={scores[k]:.4f}")

best_k = max(scores, key=scores.get)
lda = gensim.models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=best_k,
                             random_state=42, passes=15, alpha="auto")
print(f"\n=== TOPICS (k={best_k}) — paste these into your report ===")
for i, t in lda.print_topics(num_words=10):
    print(f"Topic {i}: {t}")

vocab before filter: 1411
vocab after  filter: 616
empty documents: 2
k=3  coherence=0.2906
k=4  coherence=0.2862
k=5  coherence=0.2642
k=6  coherence=0.2866

=== TOPICS (k=3) — paste these into your report ===
Topic 0: 0.071*"food" + 0.045*"service" + 0.045*"good" + 0.030*"great" + 0.023*"biryani" + 0.022*"chicken" + 0.018*"place" + 0.016*"delicious" + 0.015*"ambience" + 0.014*"time"
Topic 1: 0.041*"dosa" + 0.031*"place" + 0.022*"masala" + 0.020*"one" + 0.017*"bangalore" + 0.015*"best" + 0.013*"visited" + 0.013*"experience" + 0.013*"taste" + 0.012*"south"
Topic 2: 0.061*"food" + 0.026*"service" + 0.024*"experience" + 0.023*"place" + 0.019*"amazing" + 0.019*"quality" + 0.018*"taste" + 0.018*"poor" + 0.016*"biryani" + 0.014*"best"


### Build dictionary and corpus

In [3]:
import json
os.makedirs(MODELS_DIR, exist_ok=True)
lda.save(os.path.join(MODELS_DIR, "lda_model.gensim"))
dictionary.save(os.path.join(MODELS_DIR, "lda_dictionary.gensim"))

labels = {str(i): tag_aspects(" ".join(w for w, _ in lda.show_topic(i, 15)))[0] + f" (topic {i})"
          for i in range(best_k)}
with open(os.path.join(MODELS_DIR, "topic_labels.json"), "w") as f:
    json.dump(labels, f, indent=2)
print(json.dumps(labels, indent=2))
print("\nDone. Now run:  python verify.py   then   streamlit run app\\streamlit_app.py")

{
  "0": "Food (topic 0)",
  "1": "Food (topic 1)",
  "2": "Food (topic 2)"
}

Done. Now run:  python verify.py   then   streamlit run app\streamlit_app.py


### Train the LDA model

### Manually label each topic after reading the top words

### Assign the dominant topic to each review

### Save the model